# Эксперимент: Фонемный анализ лучшей модели (Whisper SVM) (НОВЫЙ)

**Группа:** 06_phoneme_analysis

## Мотивация

В checkpoint_3 буквы управления (Л, Р, С, Т, Ц, Ч, Ш, Щ) использовались только как  
дополнительные бинарные признаки (+8 к вектору), но никогда не анализировались отдельно.

Этот эксперимент отвечает на клинически важные вопросы:
- Насколько хорошо модель детектирует **каждый тип дефекта** отдельно?
- Какие фонемы труднее всего диагностировать?
- Есть ли записи с несколькими контрольными буквами — и влияет ли это на точность?

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score
from transformers import WhisperProcessor, WhisperModel
import matplotlib.pyplot as plt
import seaborn as sns

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix, load_dataset_csv, load_audio
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_artifact_if_exists

In [ ]:
paths_all, labels_all, df = load_dataset_csv()

base_cols = {"filename", "dir", "label", "duration", "path"}
letter_cols = [c for c in df.columns if c not in base_cols]
print(f"Буквы управления: {letter_cols}")
print(f"Всего записей: {len(df)}")
print("\nКоличество записей по буквам:")
display(df[letter_cols].sum().astype(int).to_frame("count").T)

In [ ]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()

df_test = df[df["path"].isin(paths_test)].copy().reset_index(drop=True)
path_to_idx = {p: i for i, p in enumerate(paths_test)}
df_test["_sort"] = df_test["path"].map(path_to_idx)
df_test = df_test.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

print(f"Train+Val: {len(paths_trainval)}  |  Holdout Test: {len(paths_test)}")
print(f"\nРаспределение test по классам:")
print(df_test["label"].value_counts().rename(index={0: "good", 1: "bad"}))

In [ ]:
MODEL_ID = "openai/whisper-small"
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

processor = WhisperProcessor.from_pretrained(MODEL_ID)
whisper = WhisperModel.from_pretrained(MODEL_ID).to(device)
whisper.eval()

def extract_whisper(path):
    y, _ = load_audio(path, sr=16000)
    feats = processor.feature_extractor(y, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        hs = whisper.encoder(input_features=feats.input_features.to(device)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

print("Извлечение эмбеддингов для train+val...")
X_trainval = build_feature_matrix(paths_trainval, extract_whisper, n_jobs=1)
print("Для test...")
X_test     = build_feature_matrix(paths_test,     extract_whisper, n_jobs=1)
X_trainval_full = np.hstack([X_trainval, letters_trainval])
X_test_full     = np.hstack([X_test,     letters_test])
print("Готово")

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.5, 1.0, 2.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01],
}

t0 = time.perf_counter()
grid = GridSearchCV(pipe, param_grid, cv=5,
                    scoring="f1_macro", refit=True, n_jobs=-1)
grid.fit(X_trainval_full, labels_trainval)
clf = grid.best_estimator_
print(f"best_params={grid.best_params_}")

from sklearn.model_selection import cross_val_predict
y_proba_cv = cross_val_predict(
    clf, X_trainval_full, labels_trainval, cv=5, method="predict_proba"
)[:, config.CLASS_BAD]
optimal_threshold = find_optimal_threshold(labels_trainval, y_proba_cv)
print(f"Оптимальный порог (CV-оценка на train+val): {optimal_threshold:.2f}")

train_time_sec = time.perf_counter() - t0

In [ ]:
with start_run("exp_whisper_phoneme", group="06_phoneme_analysis"):

    mlflow.log_params({
        "encoder":       MODEL_ID,
        "encoder_frozen": True,
        "aggregation":   "mean+std+max",
        "classifier":    "SVM RBF",
        "best_params":   str(grid.best_params_),
        "threshold":     optimal_threshold,
        "n_phonemes":    len(letter_cols),
    })

    y_proba_test = clf.predict_proba(X_test_full)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=optimal_threshold, verbose=True)

    # Per-phoneme analysis
    y_pred_test = (y_proba_test >= optimal_threshold).astype(int)
    df_test["y_pred"]  = y_pred_test
    df_test["y_proba"] = y_proba_test

    phoneme_results = []
    for letter in letter_cols:
        mask = df_test[letter].astype(bool)
        subset = df_test[mask]
        if len(subset) < 10:
            continue
        y_true_sub  = subset["label"].values
        y_pred_sub  = subset["y_pred"].values
        y_proba_sub = subset["y_proba"].values

        n_bad  = (y_true_sub == 1).sum()
        n_good = (y_true_sub == 0).sum()
        if n_bad == 0:
            continue

        f1_b = f1_score(y_true_sub, y_pred_sub, pos_label=config.CLASS_BAD, zero_division=0)
        acc  = accuracy_score(y_true_sub, y_pred_sub)
        try:
            from sklearn.metrics import roc_auc_score
            auc = roc_auc_score(y_true_sub, y_proba_sub) if n_bad > 0 and n_good > 0 else np.nan
        except Exception:
            auc = np.nan

        phoneme_results.append({
            "Буква": letter,
            "N записей": len(subset),
            "N bad":  n_bad,
            "N good": n_good,
            "F1-bad":  round(f1_b, 4),
            "Accuracy": round(acc, 4),
            "ROC-AUC": round(auc, 4) if not np.isnan(auc) else np.nan,
        })
        mlflow.log_metric(f"phoneme_f1_bad_{letter}", f1_b)

    df_phoneme = pd.DataFrame(phoneme_results).sort_values("F1-bad", ascending=False)
    display(df_phoneme.reset_index(drop=True))

    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    bars = ax.barh(df_phoneme["Буква"], df_phoneme["F1-bad"], color="steelblue")
    ax.axvline(test_metrics["f1_bad"], color="red", linestyle="--",
               label=f'Общий F1-bad={test_metrics["f1_bad"]:.3f}')
    ax.set_xlabel("F1-bad")
    ax.set_title("F1-bad по контрольным буквам")
    ax.legend()
    for bar, v in zip(bars, df_phoneme["F1-bad"]):
        ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=9)

    ax2 = axes[1]
    x = np.arange(len(df_phoneme))
    ax2.bar(x - 0.2, df_phoneme["N bad"],  0.4, label="bad",  color="tomato")
    ax2.bar(x + 0.2, df_phoneme["N good"], 0.4, label="good", color="seagreen")
    ax2.set_xticks(x); ax2.set_xticklabels(df_phoneme["Буква"])
    ax2.set_ylabel("Количество записей")
    ax2.set_title("Распределение классов по буквам (test)")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(exp_dir / "phoneme_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Confusion matrix по фонемам
    from sklearn.metrics import confusion_matrix
    cm_data = []
    for _, row in df_phoneme.iterrows():
        letter = row["Буква"]
        mask = df_test[letter].astype(bool)
        sub = df_test[mask]
        if len(sub) < 5:
            continue
        cm = confusion_matrix(sub["label"], sub["y_pred"], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        cm_data.append({"Буква": letter, "TN": tn, "FP": fp, "FN": fn, "TP": tp})

    df_cm = pd.DataFrame(cm_data).set_index("Буква")

    fig2, ax = plt.subplots(figsize=(8, max(4, len(df_cm) * 0.6)))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, ax=ax)
    ax.set_title("Confusion matrix по фонемам (TN|FP|FN|TP)")
    plt.tight_layout()
    plt.savefig(exp_dir / "confusion_by_phoneme.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Сохранение
    df_phoneme.to_csv(exp_dir / "phoneme_metrics.csv", index=False, encoding="utf-8")
    df_cm.to_csv(exp_dir / "phoneme_confusion.csv", encoding="utf-8")

    worst = df_phoneme.iloc[-1]
    best  = df_phoneme.iloc[0]

    save_result_csv(
        exp_dir=exp_dir,
        experiment_id="exp_whisper_phoneme",
        experiment_name="Whisper SVM — фонемный анализ",
        model="Whisper-small + SVM (per-phoneme evaluation)",
        accuracy=test_metrics["accuracy"],
        f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],
        roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"],
        recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        notes=(
            f"Per-phoneme analysis: {len(df_phoneme)} букв | "
            f"best=[{best['Буква']}] F1={best['F1-bad']:.3f} | "
            f"worst=[{worst['Буква']}] F1={worst['F1-bad']:.3f} | "
            f"thr={optimal_threshold:.2f}"
        ),
        train_time_sec=train_time_sec,
    )
    log_artifact_if_exists(exp_dir / "phoneme_analysis.png")
    log_artifact_if_exists(exp_dir / "confusion_by_phoneme.png")
    log_artifact_if_exists(exp_dir / "phoneme_metrics.csv")
    print("Результаты сохранены")

print("=" * 60)
print(f"Лучшая фонема по F1-bad : [{best['Буква']}]  F1={best['F1-bad']:.3f}  (N={best['N записей']})")
print(f"Худшая фонема по F1-bad : [{worst['Буква']}]  F1={worst['F1-bad']:.3f}  (N={worst['N записей']})")
print(f"\nОбщий F1-bad на test   : {test_metrics['f1_bad']:.3f}")
print("=" * 60)